In [45]:
import pandas as pd 


df_drift = pd.read_csv('data/drifting_longlines.csv')
# df_drift = pd.read_csv('data/fixed_gear.csv')
# df_drift = pd.read_csv('data/pole_and_line.csv')
# df_drift = pd.read_csv('data/purse_seines.csv') 
# df_drift = pd.read_csv('data/trawlers.csv')
# df_drift = pd.read_csv('data/unknown.csv')

df_drift.head()

,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source
0,1.263956e+13,1.327137e+09,232994.281250,311748.65625,8.2,230.500000,14.865583,-26.853662,-1.0,dalhousie_longliner
1,1.263956e+13,1.327137e+09,233994.265625,312410.34375,7.3,238.399994,14.863870,-26.856800,-1.0,dalhousie_longliner
2,1.263956e+13,1.327137e+09,233994.265625,312410.34375,6.8,238.899994,14.861551,-26.860649,-1.0,dalhousie_longliner
3,1.263956e+13,1.327143e+09,233994.265625,315417.37500,6.9,251.800003,14.822686,-26.865898,-1.0,dalhousie_longliner
4,1.263956e+13,1.327143e+09,233996.390625,316172.56250,6.1,231.100006,14.821825,-26.867579,-1.0,dalhousie_longliner


In [46]:
# Remover nulos
print(df_drift.shape)
df_silver = df_drift.dropna(subset=['mmsi', 'timestamp', 'distance_from_shore', 'distance_from_port',
       'speed', 'course', 'lat', 'lon', 'is_fishing'])
df_silver.shape

(13968727, 10)


(13968629, 10)

In [47]:
# Remover duplicatas mesmo com source diferente, pois o mesmo ponto pode ter sido classificado por mais de um modelo.
print(df_silver.shape)
df_silver = df_silver.drop_duplicates(subset=['mmsi', 'timestamp', 'distance_from_shore', 'distance_from_port',
       'speed', 'course', 'lat', 'lon', 'is_fishing'])
df_silver.shape

(13968629, 10)


(12745844, 10)

In [48]:
df_silver["source"].unique()

<StringArray>
['dalhousie_longliner', 'false_positives', 'gfw', 'crowd_sourced']
Length: 4, dtype: str

In [49]:
df_silver =df_silver[ df_silver["source"]!="false_positives"]
df_silver["source"].unique()

<StringArray>
['dalhousie_longliner', 'gfw', 'crowd_sourced']
Length: 3, dtype: str

In [50]:
df_silver['timestamp'] = pd.to_datetime(df_silver['timestamp'], unit='s')

In [51]:
df_silver["is_fishing"] = df_silver["is_fishing"].apply(lambda r: 1 if r>0 else r )

In [52]:
df_silver= df_silver[df_silver["is_fishing"]>=0]
df_silver.shape

(219593, 10)

In [54]:
df_silver = df_silver[df_silver["course"]<=360]
df_silver["course"] = df_silver["course"].apply(lambda r: 0 if r==360 else r)
df_silver.shape

(219593, 10)

In [55]:
df_silver["speed"].describe()

count    219593.000000
mean          5.686098
std           3.788503
min           0.000000
25%           2.600000
50%           5.700000
75%           8.700000
max         102.300003
Name: speed, dtype: float64

In [56]:
# remover aoutliners de speed, pois são pontos que provavelmente não são reais.
# calcular o limite superior usando o método do IQR
Q1 = df_silver["speed"].quantile(0.25)
Q3 = df_silver["speed"].quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR
df_silver = df_silver[df_silver["speed"] <= upper_limit]
df_silver.shape

(219510, 10)

In [ ]:
# Remover pontos fora de zonas maritimas pela lat lon:
# usar mapa mundeal para remover pontos que estão em terra, pois são pontos que provavelmente não são reais.

